# M4 — MoE Top-2, two-GPU parallel run\nNotebook chạy Dense trên GPU 0 và MoE trên GPU 1 đồng thời. Cần Kaggle T4 x2; benchmark chỉ bắt đầu sau khi cả hai tiến trình thành công.\nSau khi push bản notebook sửa này, đặt SHA mới trong cell cấu hình rồi chạy notebook từ đầu. Phiên đang chạy hiện tại không thể chuyển sang chế độ song song giữa chừng.

In [ ]:
REPO_URL = 'https://github.com/shinkbb/mini-gpt.git'\nCOMMIT_SHA = 'REPLACE_WITH_FULL_PARALLEL_M4_COMMIT_SHA'\nM1_INPUT = '/kaggle/input'\nMAX_STEPS = 1000\nDEVICE = 'cuda:0 and cuda:1'  # Dense -> cuda:0; MoE -> cuda:1\nPRECISION = 'fp16'

In [ ]:
from pathlib import Path
import re, shutil, subprocess, sys
assert re.fullmatch(r'[0-9a-fA-F]{40}', COMMIT_SHA), 'Set full M4 commit SHA after pushing'
root = Path('/kaggle/working/mini-gpt-m4')
subprocess.run(['git', 'clone', REPO_URL, str(root)], check=True)
subprocess.run(['git', 'fetch', 'origin', COMMIT_SHA], cwd=root, check=True)
subprocess.run(['git', 'checkout', '--detach', COMMIT_SHA], cwd=root, check=True)
assert subprocess.check_output(['git','rev-parse','HEAD'], cwd=root, text=True).strip().lower() == COMMIT_SHA.lower()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], cwd=root, check=True)
input_root = Path(M1_INPUT)
m1_candidates = [p for p in input_root.rglob('train_tokens.pt') if (p.parent/'validation_tokens.pt').is_file() and (p.parent/'tokenizer.json').is_file()]
assert m1_candidates, 'Attach the M1 artifact dataset (tokenizer.json + train/validation_tokens.pt)'
m1_dir = m1_candidates[0].parent
(root/'artifacts').mkdir(exist_ok=True)
for name in ('tokenizer.json','train_tokens.pt','validation_tokens.pt'): shutil.copy2(m1_dir/name, root/'artifacts'/name)
print('Source:', COMMIT_SHA, 'M1 artifacts:', m1_dir)

In [ ]:
import yaml
moe_config = yaml.safe_load((root/'configs/moe.yaml').read_text())
moe_config['training']['max_steps'] = MAX_STEPS
moe_config['training']['precision'] = PRECISION
(root/'configs/moe.yaml').write_text(yaml.safe_dump(moe_config, sort_keys=False))
dense_config = yaml.safe_load((root/'configs/dense.yaml').read_text())
dense_config['training']['max_steps'] = MAX_STEPS
dense_config['training']['precision'] = PRECISION
dense_config['training']['output_dir'] = 'runs/dense_m4_control'
(root/'configs/dense_m4.yaml').write_text(yaml.safe_dump(dense_config, sort_keys=False))
dense_ckpt = root/'runs/dense_m4_control/best.pt'
moe_last = root/'runs/moe/last.pt'

def launch(command, log_path):
    return subprocess.Popen(command, cwd=root, stdout=open(log_path, 'w'), stderr=subprocess.STDOUT)

dense_args = [sys.executable, 'train.py', '--config', 'configs/dense_m4.yaml', '--max-steps', str(MAX_STEPS), '--device', 'cuda:0']
moe_args = [sys.executable, 'train.py', '--config', 'configs/moe.yaml', '--max-steps', str(MAX_STEPS), '--device', 'cuda:1']
if dense_ckpt.exists():
    print('Reuse completed Dense checkpoint:', dense_ckpt)
    dense_process = None
else:
    dense_process = launch(dense_args, '/kaggle/working/dense_train.log')
if moe_last.exists(): moe_args += ['--resume', str(moe_last)]
moe_process = launch(moe_args, '/kaggle/working/moe_train.log')

process_results = []
for name, process, log_path in [('Dense', dense_process, '/kaggle/working/dense_train.log'),
                                ('MoE', moe_process, '/kaggle/working/moe_train.log')]:
    if process is None: continue
    return_code = process.wait()
    process_results.append((name, return_code, log_path))
    print(f'{name} process finished with exit code {return_code}; log: {log_path}')
failures = [(name, code, log) for name, code, log in process_results if code != 0]
if failures:
    for name, code, log_path in failures:
        print(f'\n--- {name} log (exit {code}) ---\n', Path(log_path).read_text()[-12000:])
    raise RuntimeError('One or more GPU training processes failed; both have now stopped.')

assert dense_ckpt.is_file() and (root/'runs/moe/best.pt').is_file(), 'Both GPU trainings must finish before benchmark.'


In [ ]:
subprocess.run([sys.executable, '-m', 'experiments.benchmark_moe', '--dense-checkpoint', 'runs/dense_m4_control/best.pt', '--moe-checkpoint', 'runs/moe/best.pt', '--validation-tokens', 'artifacts/validation_tokens.pt', '--precision', PRECISION, '--device', 'cuda:0', '--batches', '100', '--repeats', '3', '--dense-training-metrics', 'runs/dense_m4_control/metrics.jsonl', '--moe-training-metrics', 'runs/moe/metrics.jsonl', '--output', 'reports/benchmark_moe.json'], cwd=root, check=True)
print('Save Version to preserve Dense/MoE checkpoints and reports/benchmark_moe.json/.csv/.svg as Kaggle Notebook Output.')